In [2]:
# create the list of anchor and target maps 
import os 
import pandas as pd
import json
from modules.data_preparation import extract_epsg

MAIN_FOLDER = '/Users/vaienti/Library/CloudStorage/OneDrive-epfl.ch/PHD_THESIS_MATERIAL/Jerusalem_Vectors/raster_cartography_new_selection/maps_original_files/'
map_folders = os.listdir(MAIN_FOLDER) # READ THE FOLDER PATH AND GET THE LIST OF THE FOLDERS INSIDE
map_folders = [x for x in map_folders if x != '.DS_Store'] # REMOVE THE .DS_STORE FILE
anchor_list = []
target_list = []

# Collecting the map files
for folder in map_folders:
    if os.path.exists(MAIN_FOLDER + folder + '/'):
        map_files = [x for x in os.listdir(MAIN_FOLDER + folder + '/') if x != '.DS_Store']  # REMOVE THE .DS_STORE FILE
        objects_to_collect = {}  # Dictionary to store the map subfiles to collect
        year = folder.split('_')[0]
        auth = folder.split('_')[1]
        name_parts = folder.split('_')
        
        if len(name_parts) == 3:
            try:
                int(name_parts[2])
                number = name_parts[2]
                name = auth + '_' + year + '_' + number
            except:
                name = auth + '_' + name_parts[2] + '_' + year
        else:
            name = auth + '_' + year
  
        for file in map_files:
            if file.endswith(name + '.png') or file.endswith(name + '.jpg') or file.endswith(name + '.tif') or file.endswith(name + '.pdf') or file.endswith(name + '.jpeg') or file.endswith(name + '.tiff'):
                image_path = MAIN_FOLDER + folder + '/' + file
                objects_to_collect['image_path'] = image_path
                
            if file.endswith('mask.png'):
                mask_path = MAIN_FOLDER + folder + '/' + file
                if os.path.exists(mask_path):
                    objects_to_collect['mask_path'] = mask_path
                
            if file.endswith('.points'):
                file_path = MAIN_FOLDER + folder + '/' + file
                objects_to_collect['points_path'] = file_path
                first_row = pd.read_csv(file_path, nrows=1, delimiter=',')
                points = pd.read_csv(file_path, skiprows=1, delimiter=',') 
                
                if points is not None:
                    objects_to_collect['points'] = points
            
                crs_info = str(first_row.columns)
                epsg_code = extract_epsg(crs_info)  # Extract EPSG code using the new function       
                objects_to_collect['epsg'] = epsg_code
            
            objects_to_collect['folder'] = folder
            objects_to_collect['folder_path'] = MAIN_FOLDER + folder + '/'
            
                
            if file.endswith('metadata.json'):
                with open(MAIN_FOLDER + folder + '/' + file) as json_file:
                    try:
                        metadata = json.load(json_file)
                        objects_to_collect['metadata'] = metadata
                    except:
                        continue
        
        if 'points' in objects_to_collect.keys() and 'mask_path' in objects_to_collect.keys():
            # Create a 'processed' folder inside the map's folder
            # if all the objects are not None
            if [x for x in objects_to_collect.values() if x is None]:
                continue
            processed_folder = os.path.join(MAIN_FOLDER, folder, 'processed')
            os.makedirs(processed_folder, exist_ok=True)  # Create the processed folder if it doesn't exist

            # Determine the output path for the processed tensor
            processed_image_name = os.path.basename(objects_to_collect['image_path']).split('.')[0] + '_processed.pt'
            processed_image_path = os.path.join(processed_folder, processed_image_name)

            anchor_list.append(objects_to_collect)
        elif 'points' not in objects_to_collect.keys() and 'image_path' in objects_to_collect.keys():
            target_list.append(objects_to_collect)

print(f'{len(anchor_list)} anchor maps out of {len(map_folders)}')
print(f'{len(target_list)} target maps out of {len(map_folders)}')


# save them with pickle
import pickle
with open('input/anchor_maps.pkl', 'wb') as f:
    pickle.dump(anchor_list, f)

with open('input/target_maps.pkl', 'wb') as f:
    pickle.dump(target_list, f)

anchor_maps = anchor_list
target_maps = target_list


Error extracting EPSG code: list index out of range, Index(['#CRS: '], dtype='object')
86 anchor maps out of 202
113 target maps out of 202


In [ ]:

anchor_image_paths = []
anchor_mask_paths = []
anchor_points_paths = []
for anchor in anchor_maps:
    anchor_image_paths.append(anchor['image_path'])
    anchor_mask_paths.append(anchor['mask_path'])
    anchor_points_paths.append(anchor['points_path'])

target_image_paths = []
target_mask_paths = []
output_dirs = []
for target in target_maps[12::]:
    target_image_paths.append(target['image_path'])
    target_mask_paths.append(target['mask_path'])
    # create a output directory for the target map
    output_dir = os.path.join(target['folder_path'], 'propagated_gcp_results')
    output_dirs.append(output_dir)
    os.makedirs(output_dir, exist_ok=True)

config = {
    "target_image_paths" : target_image_paths,
    "target_image_mask_paths" : target_mask_paths,
    "anchor_image_paths" : anchor_image_paths,
    "anchor_image_masks" : anchor_mask_paths,
    "anchor_points_paths" : anchor_points_paths,
    "output_dirs" : output_dirs
}

with open('input/config.json', 'w') as f:
    json.dump(config, f)


